# 07 - Reuters 5000 Dış Testi

Bu notebook **önce dataset'i yükler ve kontrol ettirir**, model/torch importları daha sonraki hücrelerde başlar.

Test seti:

`db/annotations/reuters_5000`

Desteklenen batch formatları:

- `REUTERS_annotation_batch_001.xlsx`
- `reuters_annotation_batch_001.xlsx`
- `.xlsx` veya `.csv`
- Dosyalar doğrudan ana klasörde veya alt klasörlerde olabilir.

Değerlendirilecek modeller:

- `original_finbert`: Dosyadaki mevcut `finbert_label` üzerinden baseline olarak değerlendirilir.
- `roberta_base`: `checkpoints/financial_sentiment_multi_model/roberta_base/final_model`
- `bert_base_uncased`: `checkpoints/financial_sentiment_multi_model/bert_base_uncased/final_model`
- `distilbert_base_uncased`: `checkpoints/financial_sentiment_multi_model/distilbert_base_uncased/final_model`

Bu notebook sonuçları dosyaya kaydetmez; sadece ekrana basar.


In [1]:
from thesis_utils import APP_ROOT, DATA_ROOT, PREVIEW_ROWS, PROJECT_ROOT, paths
# ============================================================
# CELL 1 - DATASET IMPORTLARI VE AYARLAR
# Bu hücrede torch/model yükleme yoktur.
# ============================================================

from pathlib import Path
import re
import gc
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 180)

PROJECT_DIR = PROJECT_ROOT
ANNOTATION_DIR = paths.REUTERS_ANNOTATION_DIR
CHECKPOINT_ROOT = paths.MODEL_CHECKPOINT_ROOT

VALID_LABELS = ["negative", "neutral", "positive"]

LABEL2ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

ID2LABEL = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

MODEL_RUNS = [
    {
        "run_name": "roberta_base",
        "model_dir": CHECKPOINT_ROOT / "roberta_base" / "final_model",
    },
    {
        "run_name": "bert_base_uncased",
        "model_dir": CHECKPOINT_ROOT / "bert_base_uncased" / "final_model",
    },
    {
        "run_name": "distilbert_base_uncased",
        "model_dir": CHECKPOINT_ROOT / "distilbert_base_uncased" / "final_model",
    },
]

MAX_LENGTH = 128
BATCH_SIZE = 32
TEST_SET_NAME = "reuters_annotation_5000"

print("PROJECT_DIR:", PROJECT_DIR)
print("ANNOTATION_DIR:", ANNOTATION_DIR)
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)

if not ANNOTATION_DIR.exists():
    raise FileNotFoundError(f"Annotation klasörü bulunamadı: {ANNOTATION_DIR}")


PROJECT_DIR: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis
ANNOTATION_DIR: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000
CHECKPOINT_ROOT: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model


In [2]:
# ============================================================
# CELL 2 - REUTERS BATCH DOSYALARINI OKU VE TEK DF YAP
# Model yükleme yoktur.
# ============================================================

def extract_batch_no(path):
    path = Path(path)
    m = re.search(r"annotation[_\s-]*batch[_\s-]*(\d+)", path.stem.lower())
    if m:
        return int(m.group(1))
    m = re.search(r"batch[_\s-]*(\d+)", path.stem.lower())
    if m:
        return int(m.group(1))
    return 999999


def normalize_colname(c):
    c = str(c).strip()
    c = re.sub(r"\s+", "_", c)
    return c


def read_batch_file(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".xlsx":
        df = pd.read_excel(path)
    elif suffix == ".csv":
        df = pd.read_csv(path, encoding="utf-8-sig")
    else:
        raise ValueError(f"Desteklenmeyen dosya türü: {path}")

    df.columns = [normalize_colname(c) for c in df.columns]
    df = df.dropna(how="all").copy()

    # Gereksiz kolonları temizle
    drop_cols = []
    for c in df.columns:
        lc = str(c).lower().strip()
        if lc.startswith("unnamed"):
            drop_cols.append(c)
        elif c in ["Özet", "Değer", "Ozet", "Deger", "Field", "Value"]:
            drop_cols.append(c)

    df = df.drop(columns=drop_cols, errors="ignore")

    df["source_file"] = path.name
    df["source_path"] = str(path)
    df["batch_no"] = extract_batch_no(path)

    return df


# Ana klasör ve alt klasörlerde batch ara
patterns = [
    "*annotation_batch_*.xlsx",
    "*annotation_batch_*.csv",
    "*ANNOTATION_BATCH_*.xlsx",
    "*ANNOTATION_BATCH_*.csv",
]

batch_files = []
for pat in patterns:
    batch_files.extend(ANNOTATION_DIR.rglob(pat))

# Tekilleştir ve geçici Excel dosyalarını çıkar
batch_files = sorted(
    {p for p in batch_files if p.is_file() and not p.name.startswith("~$")},
    key=lambda p: (extract_batch_no(p), p.name.lower())
)

if not batch_files:
    print("Klasörde bulunan dosyalar:")
    for p in sorted(ANNOTATION_DIR.rglob("*")):
        if p.is_file():
            print("-", p)
    raise FileNotFoundError(f"Batch dosyası bulunamadı: {ANNOTATION_DIR}")

print("Okunacak batch sayısı:", len(batch_files))
print("İlk 3 dosya:")
for p in batch_files[:PREVIEW_ROWS]:
    print("-", extract_batch_no(p), p)
print("Son 3 dosya:")
for p in batch_files[-PREVIEW_ROWS:]:
    print("-", extract_batch_no(p), p)

all_parts = []

for p in batch_files:
    temp = read_batch_file(p)
    all_parts.append(temp)

annotation_all_df = pd.concat(all_parts, ignore_index=True)

# annotation_id boş satırları çıkar
if "annotation_id" in annotation_all_df.columns:
    annotation_all_df = annotation_all_df[annotation_all_df["annotation_id"].notna()].copy()

# String kolonları temizle
for col in [
    "annotation_id",
    "source_dataset",
    "text_en",
    "text_tr",
    "finbert_label",
    "chatgpt_label",
    "chatgpt_confidence",
    "chatgpt_reason_tr",
    "final_label",
    "annotation_note",
    "finbert_correctness",
    "finbert_correctness_note",
]:
    if col in annotation_all_df.columns:
        annotation_all_df[col] = annotation_all_df[col].astype("string").str.strip()

if "date" in annotation_all_df.columns:
    annotation_all_df["date"] = pd.to_datetime(annotation_all_df["date"], errors="coerce")

if "finbert_confidence" in annotation_all_df.columns:
    annotation_all_df["finbert_confidence"] = pd.to_numeric(annotation_all_df["finbert_confidence"], errors="coerce")

if "finbert_score" in annotation_all_df.columns:
    annotation_all_df["finbert_score"] = pd.to_numeric(annotation_all_df["finbert_score"], errors="coerce")

# annotation_id sırasına göre sırala
def extract_ann_no(x):
    m = re.search(r"(\d+)", str(x))
    if m:
        return int(m.group(1))
    return 999999999

if "annotation_id" in annotation_all_df.columns:
    annotation_all_df["_ann_no"] = annotation_all_df["annotation_id"].apply(extract_ann_no)
    annotation_all_df = annotation_all_df.sort_values(["_ann_no", "batch_no"]).drop(columns=["_ann_no"]).reset_index(drop=True)
else:
    annotation_all_df = annotation_all_df.reset_index(drop=True)

print("\nannotation_all_df shape:", annotation_all_df.shape)
print("\nColumns:")
print(annotation_all_df.columns.tolist())

print("\nBatch dağılımı:")
print(annotation_all_df["batch_no"].value_counts().sort_index().head(PREVIEW_ROWS))
print("...")
print(annotation_all_df["batch_no"].value_counts().sort_index().tail(PREVIEW_ROWS))




Okunacak batch sayısı: 200
İlk 3 dosya:
- 1 D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\annotation_batches_50\reuters_annotation_batch_001.csv
- 1 D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\REUTERS_annotation_batch_001.xlsx
- 2 D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\annotation_batches_50\reuters_annotation_batch_002.csv
Son 3 dosya:
- 99 D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\REUTERS_annotation_batch_099.xlsx
- 100 D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\annotation_batches_50\reuters_annotation_batch_100.csv
- 100 D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\REUTERS_annotation_batch_100.xlsx

annotation_all_df shape: (9900, 47)

Columns:
['annotati

In [3]:
# ============================================================
# CELL 2 - REUTERS BATCH DOSYALARINI ROBUST OKU VE TEK DF YAP
# Model yükleme yoktur.
# Farklı kolon/header yapısı olan batchleri düzeltir.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

# ------------------------------------------------------------
# Yardımcı fonksiyonlar
# ------------------------------------------------------------

def extract_batch_no(path):
    path = Path(path)
    m = re.search(r"annotation[_\s-]*batch[_\s-]*(\d+)", path.stem.lower())
    if m:
        return int(m.group(1))
    m = re.search(r"batch[_\s-]*(\d+)", path.stem.lower())
    if m:
        return int(m.group(1))
    return 999999


def normalize_colname(c):
    c = str(c).strip()
    c = re.sub(r"\s+", "_", c)
    c = c.replace("\n", "_").replace("\r", "_")
    return c


def make_unique_columns(cols):
    seen = {}
    out = []

    for col in cols:
        col = normalize_colname(col)

        if col == "" or col.lower() in ["nan", "none", "<na>"]:
            col = "unnamed"

        if col not in seen:
            seen[col] = 0
            out.append(col)
        else:
            seen[col] += 1
            out.append(f"{col}_{seen[col]}")

    return out


def clean_string_series(s):
    return (
        s.astype("string")
        .str.strip()
        .replace(["", "nan", "NaN", "None", "none", "<NA>", "<na>"], pd.NA)
    )


def read_raw_no_header(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".xlsx":
        return pd.read_excel(path, header=None, dtype=object)
    elif suffix == ".csv":
        return pd.read_csv(path, header=None, encoding="utf-8-sig", dtype=object)
    else:
        raise ValueError(f"Desteklenmeyen dosya türü: {path}")


def find_header_row(raw_df):
    """
    Dosya içinde annotation_id geçen satırı header kabul eder.
    Böylece Batch 005 / 018 gibi üstte açıklama satırı olan dosyalar düzelir.
    """
    for idx in range(len(raw_df)):
        row_values = (
            raw_df.iloc[idx]
            .astype(str)
            .str.strip()
            .str.lower()
            .tolist()
        )

        if "annotation_id" in row_values:
            return idx

    return None


def read_batch_file_robust(path):
    path = Path(path)
    batch_no = extract_batch_no(path)

    raw = read_raw_no_header(path)
    raw = raw.dropna(how="all").copy()

    header_idx = find_header_row(raw)

    if header_idx is None:
        # Son çare: normal oku
        if path.suffix.lower() == ".xlsx":
            df = pd.read_excel(path, dtype=object)
        else:
            df = pd.read_csv(path, encoding="utf-8-sig", dtype=object)

        df.columns = make_unique_columns(df.columns)
    else:
        header = raw.iloc[header_idx].tolist()
        data = raw.iloc[header_idx + 1:].copy()
        data.columns = make_unique_columns(header)
        df = data

    df = df.dropna(how="all").copy()
    df = df.dropna(axis=1, how="all").copy()

    # Kolon isimlerini normalize et
    df.columns = make_unique_columns(df.columns)

    # Gereksiz kolonları temizle
    drop_cols = []

    for c in df.columns:
        lc = str(c).lower().strip()

        if lc.startswith("unnamed"):
            drop_cols.append(c)
        elif lc in ["özet", "değer", "ozet", "deger", "field", "value"]:
            drop_cols.append(c)
        elif "reuters_annotation_batch" in lc and c != "annotation_id":
            drop_cols.append(c)
        elif "reuters_annotation" in lc and c != "annotation_id":
            drop_cols.append(c)

    df = df.drop(columns=drop_cols, errors="ignore").copy()

    # annotation_id yoksa bu dosya sorunludur
    if "annotation_id" not in df.columns:
        df["annotation_id"] = pd.NA

    df["annotation_id"] = clean_string_series(df["annotation_id"])
    df = df[df["annotation_id"].notna()].copy()

    # --------------------------------------------------------
    # text_en standardizasyonu
    # Bazı dosyalarda text, bazılarında text_en, bazılarında source_text var.
    # Hepsi text_en altında birleşecek.
    # --------------------------------------------------------
    if "text_en" not in df.columns:
        df["text_en"] = pd.NA

    df["text_en"] = clean_string_series(df["text_en"])

    fallback_text_cols = ["text", "source_text", "headline", "title"]

    for col in fallback_text_cols:
        if col in df.columns:
            fallback = clean_string_series(df[col])
            df["text_en"] = df["text_en"].fillna(fallback)

    # --------------------------------------------------------
    # Label kolonları
    # --------------------------------------------------------
    if "chatgpt_label" not in df.columns:
        df["chatgpt_label"] = pd.NA

    if "final_label" not in df.columns:
        df["final_label"] = pd.NA

    df["chatgpt_label"] = clean_string_series(df["chatgpt_label"]).str.lower()
    df["final_label"] = clean_string_series(df["final_label"]).str.lower()

    # final_label boşsa chatgpt_label ile doldur
    df["final_label"] = df["final_label"].fillna(df["chatgpt_label"])

    if "finbert_label" not in df.columns:
        df["finbert_label"] = pd.NA

    df["finbert_label"] = clean_string_series(df["finbert_label"]).str.lower()

    # --------------------------------------------------------
    # Numeric / date kolonları
    # --------------------------------------------------------
    if "finbert_score" in df.columns:
        df["finbert_score"] = pd.to_numeric(df["finbert_score"], errors="coerce")
    else:
        df["finbert_score"] = np.nan

    if "finbert_confidence" in df.columns:
        df["finbert_confidence"] = pd.to_numeric(df["finbert_confidence"], errors="coerce")
    else:
        df["finbert_confidence"] = np.nan

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    else:
        df["date"] = pd.NaT

    # --------------------------------------------------------
    # Eksik standart kolonları ekle
    # --------------------------------------------------------
    canonical_cols = [
        "batch_id",
        "annotation_id",
        "date",
        "finbert_score",
        "finbert_label",
        "finbert_confidence",
        "text_en",
        "text_tr",
        "chatgpt_label",
        "chatgpt_score",
        "chatgpt_confidence",
        "chatgpt_reason_tr",
        "final_label",
        "finbert_correctness",
        "review_needed",
        "row_note",
        "annotation_note",
        "review_note_tr",
        "review_note",
        "review_flag",
        "finbert_comparison",
        "finbert_match",
        "finbert_vs_chatgpt",
        "review_priority",
        "needs_review",
        "finbert_alignment",
        "source_batch",
        "row_no",
    ]

    for col in canonical_cols:
        if col not in df.columns:
            df[col] = pd.NA

    df["source_file"] = path.name
    df["source_path"] = str(path)
    df["batch_no"] = batch_no

    # text_en son temizlik
    df["text_en"] = clean_string_series(df["text_en"])

    # Kolon sırası
    extra_cols = [
        c for c in df.columns
        if c not in canonical_cols + ["source_file", "source_path", "batch_no"]
    ]

    df = df[canonical_cols + ["source_file", "source_path", "batch_no"] + extra_cols].copy()

    return df


# ------------------------------------------------------------
# Batch dosyalarını bul
# ------------------------------------------------------------

patterns = [
    "*annotation_batch_*.xlsx",
    "*annotation_batch_*.csv",
    "*ANNOTATION_BATCH_*.xlsx",
    "*ANNOTATION_BATCH_*.csv",
]

batch_files = []

for pat in patterns:
    batch_files.extend(ANNOTATION_DIR.rglob(pat))

batch_files = sorted(
    {p for p in batch_files if p.is_file() and not p.name.startswith("~$")},
    key=lambda p: (extract_batch_no(p), p.name.lower())
)

if not batch_files:
    print("Klasörde bulunan dosyalar:")
    for p in sorted(ANNOTATION_DIR.rglob("*")):
        if p.is_file():
            print("-", p)
    raise FileNotFoundError(f"Batch dosyası bulunamadı: {ANNOTATION_DIR}")

print("Okunacak batch sayısı:", len(batch_files))

print("\nİlk 3 dosya:")
for p in batch_files[:PREVIEW_ROWS]:
    print("-", extract_batch_no(p), p)

print("\nSon 3 dosya:")
for p in batch_files[-PREVIEW_ROWS:]:
    print("-", extract_batch_no(p), p)


# ------------------------------------------------------------
# Tüm batchleri oku
# ------------------------------------------------------------

all_parts = []
read_reports = []

for p in batch_files:
    batch_no = extract_batch_no(p)

    try:
        temp = read_batch_file_robust(p)

        all_parts.append(temp)

        read_reports.append({
            "batch_no": batch_no,
            "file_name": p.name,
            "rows": len(temp),
            "text_en_missing": temp["text_en"].isna().sum(),
            "finbert_label_missing": temp["finbert_label"].isna().sum(),
            "chatgpt_label_missing": temp["chatgpt_label"].isna().sum(),
            "final_label_missing": temp["final_label"].isna().sum(),
            "status": "OK",
        })

    except Exception as e:
        read_reports.append({
            "batch_no": batch_no,
            "file_name": p.name,
            "rows": None,
            "text_en_missing": None,
            "finbert_label_missing": None,
            "chatgpt_label_missing": None,
            "final_label_missing": None,
            "status": f"ERROR: {e}",
        })

read_report_df = pd.DataFrame(read_reports).sort_values("batch_no").reset_index(drop=True)

print("\nOkuma raporu:")
display(read_report_df)

if not all_parts:
    raise ValueError("Hiç batch okunamadı.")

annotation_all_df = pd.concat(all_parts, ignore_index=True)

# ------------------------------------------------------------
# Son temizlik
# ------------------------------------------------------------

annotation_all_df["annotation_id"] = clean_string_series(annotation_all_df["annotation_id"])
annotation_all_df["text_en"] = clean_string_series(annotation_all_df["text_en"])

annotation_all_df = annotation_all_df[
    annotation_all_df["annotation_id"].notna()
].copy()

for col in ["finbert_label", "chatgpt_label", "final_label"]:
    annotation_all_df[col] = (
        annotation_all_df[col]
        .astype("string")
        .str.lower()
        .str.strip()
        .replace(["", "nan", "none", "<NA>", "<na>"], pd.NA)
    )

annotation_all_df["final_label"] = annotation_all_df["final_label"].fillna(
    annotation_all_df["chatgpt_label"]
)

annotation_all_df["finbert_confidence"] = pd.to_numeric(
    annotation_all_df["finbert_confidence"],
    errors="coerce"
)

annotation_all_df["finbert_score"] = pd.to_numeric(
    annotation_all_df["finbert_score"],
    errors="coerce"
)

annotation_all_df["date"] = pd.to_datetime(
    annotation_all_df["date"],
    errors="coerce"
)

# annotation_id sırasına göre sırala
def extract_ann_no(x):
    m = re.search(r"(\d+)", str(x))
    if m:
        return int(m.group(1))
    return 999999999

annotation_all_df["_ann_no"] = annotation_all_df["annotation_id"].apply(extract_ann_no)

annotation_all_df = (
    annotation_all_df
    .sort_values(["_ann_no", "batch_no"])
    .drop(columns=["_ann_no"])
    .reset_index(drop=True)
)

annotation_all_df["n_words"] = annotation_all_df["text_en"].str.split().str.len()
annotation_all_df["text_len"] = annotation_all_df["text_en"].str.len()

# ------------------------------------------------------------
# Kontrol çıktıları
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("FINAL MERGED annotation_all_df")
print("=" * 120)

print("annotation_all_df shape:", annotation_all_df.shape)

print("\nBoşluk kontrolü:")
for col in ["annotation_id", "text_en", "finbert_label", "chatgpt_label", "final_label"]:
    print(col, "boş:", annotation_all_df[col].isna().sum())

print("\nBatch dağılımı:")
print(annotation_all_df["batch_no"].value_counts().sort_index().head(PREVIEW_ROWS))
print("...")
print(annotation_all_df["batch_no"].value_counts().sort_index().tail(PREVIEW_ROWS))

print("\nEksik batch var mı?")
all_batch_nos = set(range(1, 101))
found_batch_nos = set(annotation_all_df["batch_no"].dropna().astype(int).unique())
missing_batches = sorted(all_batch_nos - found_batch_nos)
print("Eksik batchler:", missing_batches)

print("\nSatır sayısı 50 olmayan batchler:")
batch_count_df = annotation_all_df["batch_no"].value_counts().sort_index().reset_index()
batch_count_df.columns = ["batch_no", "row_count"]
display(batch_count_df[batch_count_df["row_count"] != 50])

print("\nFinBERT label dağılımı:")
print(annotation_all_df["finbert_label"].value_counts(dropna=False))

print("\nChatGPT label dağılımı:")
print(annotation_all_df["chatgpt_label"].value_counts(dropna=False))

print("\nFinal label dağılımı:")
print(annotation_all_df["final_label"].value_counts(dropna=False))

valid_labels = ["negative", "neutral", "positive"]

eval_mask = (
    annotation_all_df["text_en"].notna()
    & (annotation_all_df["text_en"] != "")
    & annotation_all_df["final_label"].isin(valid_labels)
)

print("\nModel testine girecek geçerli satır sayısı:", eval_mask.sum())

print("\nSorunlu okunan batchler:")
display(
    read_report_df[
        (read_report_df["status"] != "OK")
        | (read_report_df["rows"] != 50)
        | (read_report_df["text_en_missing"].fillna(999999) > 0)
    ]
)

#annotation_all_df

Okunacak batch sayısı: 200

İlk 3 dosya:
- 1 D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\annotation_batches_50\reuters_annotation_batch_001.csv
- 1 D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\REUTERS_annotation_batch_001.xlsx
- 2 D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\annotation_batches_50\reuters_annotation_batch_002.csv

Son 3 dosya:
- 99 D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\REUTERS_annotation_batch_099.xlsx
- 100 D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\annotation_batches_50\reuters_annotation_batch_100.csv
- 100 D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\REUTERS_annotation_batch_100.xlsx

Okuma raporu:


,batch_no,file_name,rows,text_en_missing,finbert_label_missing,chatgpt_label_missing,final_label_missing,status
0,1,reuters_annotation_batch_001.csv,50,0,0,50,50,OK
1,1,REUTERS_annotation_batch_001.xlsx,50,0,0,0,0,OK
2,2,reuters_annotation_batch_002.csv,50,0,0,50,50,OK
3,2,REUTERS_annotation_batch_002.xlsx,50,0,0,0,0,OK
4,3,reuters_annotation_batch_003.csv,50,0,0,50,50,OK
...,...,...,...,...,...,...,...,...
195,98,REUTERS_annotation_batch_098.xlsx,50,0,0,0,0,OK
196,99,reuters_annotation_batch_099.csv,50,0,0,50,50,OK
197,99,REUTERS_annotation_batch_099.xlsx,50,0,0,0,0,OK
198,100,reuters_annotation_batch_100.csv,50,0,0,50,50,OK



FINAL MERGED annotation_all_df
annotation_all_df shape: (10000, 44)

Boşluk kontrolü:
annotation_id boş: 0
text_en boş: 0
finbert_label boş: 0
chatgpt_label boş: 5000
final_label boş: 5000

Batch dağılımı:
batch_no
1    100
2    100
3    100
Name: count, dtype: int64
...
batch_no
98     100
99     100
100    100
Name: count, dtype: int64

Eksik batch var mı?
Eksik batchler: []

Satır sayısı 50 olmayan batchler:


,batch_no,row_count
0,1,100
1,2,100
2,3,100
3,4,100
4,5,100
...,...,...
95,96,100
96,97,100
97,98,100
98,99,100



FinBERT label dağılımı:
finbert_label
positive    3494
negative    3492
neutral     3014
Name: count, dtype: int64[pyarrow]

ChatGPT label dağılımı:
chatgpt_label
<NA>        5000
positive    2275
negative    1579
neutral     1146
Name: count, dtype: int64[pyarrow]

Final label dağılımı:
final_label
<NA>        5000
positive    2275
negative    1579
neutral     1146
Name: count, dtype: int64[pyarrow]

Model testine girecek geçerli satır sayısı: 5000

Sorunlu okunan batchler:


,batch_no,file_name,rows,text_en_missing,finbert_label_missing,chatgpt_label_missing,final_label_missing,status


In [4]:
# ============================================================
# CELL 2C - annotation_all_df TEMİZLE / KOLONLARI BİRLEŞTİR
# Gereksiz kolonları atar, tek temiz df oluşturur.
# Model yükleme yoktur.
# ============================================================

import pandas as pd
import numpy as np
import re

if "annotation_all_df" not in globals():
    raise ValueError("annotation_all_df bulunamadı. Önce batch birleştirme hücresini çalıştır.")

df = annotation_all_df.copy()

# ------------------------------------------------------------
# 1) Yardımcı fonksiyon
# ------------------------------------------------------------
def clean_str_col(s):
    return (
        s.astype("string")
        .str.strip()
        .replace(["", "nan", "NaN", "None", "none", "<NA>", "<na>"], pd.NA)
    )

# ------------------------------------------------------------
# 2) Metin kolonlarını tek text_en altında birleştir
# Öncelik:
# text_en -> text -> source_text
# ------------------------------------------------------------
if "text_en" not in df.columns:
    df["text_en"] = pd.NA

df["text_en"] = clean_str_col(df["text_en"])

for fallback_col in ["text", "source_text", "headline", "title"]:
    if fallback_col in df.columns:
        df[fallback_col] = clean_str_col(df[fallback_col])
        df["text_en"] = df["text_en"].fillna(df[fallback_col])

# ------------------------------------------------------------
# 3) Label kolonlarını temizle
# final_label boşsa chatgpt_label kullan
# ------------------------------------------------------------
for col in ["finbert_label", "chatgpt_label", "final_label"]:
    if col not in df.columns:
        df[col] = pd.NA

    df[col] = (
        clean_str_col(df[col])
        .str.lower()
        .str.strip()
    )

valid_labels = ["negative", "neutral", "positive"]

df["final_label"] = df["final_label"].fillna(df["chatgpt_label"])

# Geçersiz label varsa NaN yap
for col in ["finbert_label", "chatgpt_label", "final_label"]:
    df.loc[~df[col].isin(valid_labels), col] = pd.NA

# ------------------------------------------------------------
# 4) Sayısal kolonları düzelt
# ------------------------------------------------------------
if "finbert_score" in df.columns:
    df["finbert_score"] = pd.to_numeric(df["finbert_score"], errors="coerce")
else:
    df["finbert_score"] = np.nan

if "finbert_confidence" in df.columns:
    df["finbert_confidence"] = pd.to_numeric(df["finbert_confidence"], errors="coerce")
else:
    df["finbert_confidence"] = np.nan

if "chatgpt_score" in df.columns:
    df["chatgpt_score"] = pd.to_numeric(df["chatgpt_score"], errors="coerce")
else:
    df["chatgpt_score"] = np.nan

# ------------------------------------------------------------
# 5) Tarih düzelt
# ------------------------------------------------------------
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
else:
    df["date"] = pd.NaT

# ------------------------------------------------------------
# 6) Annotation ID temizle
# ------------------------------------------------------------
if "annotation_id" not in df.columns:
    raise ValueError("annotation_id kolonu yok.")

df["annotation_id"] = clean_str_col(df["annotation_id"])

df = df[df["annotation_id"].notna()].copy()

# ------------------------------------------------------------
# 7) Türkçe metin / açıklama kolonları
# ------------------------------------------------------------
for col in [
    "text_tr",
    "chatgpt_confidence",
    "chatgpt_reason_tr",
    "finbert_correctness",
    "review_needed",
    "row_note",
]:
    if col not in df.columns:
        df[col] = pd.NA
    df[col] = clean_str_col(df[col])

# ------------------------------------------------------------
# 8) Temiz canonical kolonları seç
# ------------------------------------------------------------
clean_cols = [
    "annotation_id",
    "date",
    "text_en",
    "text_tr",
    "finbert_score",
    "finbert_label",
    "finbert_confidence",
    "chatgpt_label",
    "chatgpt_score",
    "chatgpt_confidence",
    "chatgpt_reason_tr",
    "final_label",
    "finbert_correctness",
    "review_needed",
    "row_note",
    "source_file",
    "source_path",
    "batch_no",
]

# Eksik varsa ekle
for col in clean_cols:
    if col not in df.columns:
        df[col] = pd.NA

clean_df = df[clean_cols].copy()

# ------------------------------------------------------------
# 9) Sıralama
# ------------------------------------------------------------
def extract_ann_no(x):
    m = re.search(r"(\d+)", str(x))
    if m:
        return int(m.group(1))
    return 999999999

clean_df["_ann_no"] = clean_df["annotation_id"].apply(extract_ann_no)

clean_df = (
    clean_df
    .sort_values(["_ann_no", "batch_no"])
    .drop(columns=["_ann_no"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 10) Ek kontrol kolonları
# ------------------------------------------------------------
clean_df["n_words"] = clean_df["text_en"].str.split().str.len()
clean_df["text_len"] = clean_df["text_en"].str.len()

# ------------------------------------------------------------
# 11) Duplicate kontrol
# ------------------------------------------------------------
dup_ann = clean_df["annotation_id"].duplicated().sum()
dup_text = clean_df["text_en"].duplicated().sum()

print("=" * 120)
print("CLEAN REUTERS ANNOTATION DF READY")
print("=" * 120)

print("Shape:", clean_df.shape)
print("Duplicate annotation_id:", dup_ann)
print("Duplicate text_en:", dup_text)

print("\nBoşluk kontrolü:")
for col in ["annotation_id", "text_en", "finbert_label", "chatgpt_label", "final_label"]:
    print(col, "boş:", clean_df[col].isna().sum())

print("\nBatch dağılımı:")
print(clean_df["batch_no"].value_counts().sort_index().head(PREVIEW_ROWS))
print("...")
print(clean_df["batch_no"].value_counts().sort_index().tail(PREVIEW_ROWS))

print("\nFinBERT label dağılımı:")
print(clean_df["finbert_label"].value_counts(dropna=False))

print("\nChatGPT label dağılımı:")
print(clean_df["chatgpt_label"].value_counts(dropna=False))

print("\nFinal label dağılımı:")
print(clean_df["final_label"].value_counts(dropna=False))

valid_eval_mask = (
    clean_df["text_en"].notna()
    & (clean_df["text_en"] != "")
    & clean_df["final_label"].isin(valid_labels)
)

print("\nModel testine girecek satır sayısı:", valid_eval_mask.sum())


# Bundan sonra model testlerinde bunu kullanacağız
annotation_all_df = clean_df.copy()

CLEAN REUTERS ANNOTATION DF READY
Shape: (10000, 20)
Duplicate annotation_id: 5000
Duplicate text_en: 5000

Boşluk kontrolü:
annotation_id boş: 0
text_en boş: 0
finbert_label boş: 0
chatgpt_label boş: 5000
final_label boş: 5000

Batch dağılımı:
batch_no
1    100
2    100
3    100
Name: count, dtype: int64
...
batch_no
98     100
99     100
100    100
Name: count, dtype: int64

FinBERT label dağılımı:
finbert_label
positive    3494
negative    3492
neutral     3014
Name: count, dtype: int64[pyarrow]

ChatGPT label dağılımı:
chatgpt_label
<NA>        5000
positive    2275
negative    1579
neutral     1146
Name: count, dtype: int64[pyarrow]

Final label dağılımı:
final_label
<NA>        5000
positive    2275
negative    1579
neutral     1146
Name: count, dtype: int64[pyarrow]

Model testine girecek satır sayısı: 5000


In [5]:
annotation_all_df = annotation_all_df.drop(
    columns=["source_file", "source_path"],
    errors="ignore"
)

annotation_all_df

,annotation_id,date,text_en,text_tr,finbert_score,finbert_label,finbert_confidence,chatgpt_label,chatgpt_score,chatgpt_confidence,chatgpt_reason_tr,final_label,finbert_correctness,review_needed,row_note,batch_no,n_words,text_len
0,REUTERS_ANN_00001,2007-04-12,s.africa watchdog to probe gold fields bid report,<NA>,-1,negative,0.683211,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,8,49
1,REUTERS_ANN_00001,2007-04-12,s.africa watchdog to probe gold fields bid report,Güney Afrika rekabet kurumu Gold Fields teklifi raporunu inceleyecek,-1,negative,0.683211,negative,-1.0,0.86,Bir satın alma teklifinin rekabet kurumu tarafından incelenmesi düzenleyici belirsizlik ve işlem riski yaratır.,negative,same,no,FinBERT etiketi bağlamla uyumlu; 'watchdog to probe' ifadesi düzenleyici soruşturma/inceleme riskini negatif yakalamış.,1,8,49
2,REUTERS_ANN_00002,2007-04-26,new barbie girls sashay into view with mp-3,<NA>,0,neutral,0.847032,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,8,43
3,REUTERS_ANN_00002,2007-04-26,new barbie girls sashay into view with mp-3,Yeni Barbie Girls MP3 ile sahneye çıkıyor,0,neutral,0.847032,positive,1.0,0.76,Yeni MP3 özellikli ürün lansmanı ürün yeniliği ve satış potansiyeli açısından sınırlı olumlu sinyal verir.,positive,different,yes,FinBERT bunu nötr görmüş olabilir çünkü başlık finansal metrik içermiyor; ancak yeni ürün tanıtımı pozitif ürün/growth sinyalidir.,1,8,43
4,REUTERS_ANN_00003,2006-12-17,"stocks await data, mergers and santa",<NA>,0,neutral,0.865116,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,6,36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,REUTERS_ANN_04998,2007-01-30,icahn seeks motorola to return cash to shareholders,"Icahn, Motorola’nın hissedarlara nakit iade etmesini istiyor",0,neutral,0.816541,positive,1.0,0.84,Hissedarlara nakit iadesi talebi hissedar getirisi ve aktivist değer yaratma sinyali verir.,positive,different,yes,FinBERT bunu nötr görmüş olabilir çünkü talep henüz gerçekleşmemiş; ancak 'return cash to shareholders' pozitif hissedar değeri sinyalidir.,100,8,51
9996,REUTERS_ANN_04999,2007-01-31,hilton hotels posts higher 4th-quarter profit,<NA>,1,positive,0.952278,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,100,6,45
9997,REUTERS_ANN_04999,2007-01-31,hilton hotels posts higher 4th-quarter profit,Hilton Hotels daha yüksek dördüncü çeyrek kârı açıkladı,1,positive,0.952278,positive,1.0,0.98,Daha yüksek kâr doğrudan olumlu finansal performans göstergesidir.,positive,same,no,FinBERT etiketi bağlamla uyumlu; 'higher profit' net pozitif.,100,6,45
9998,REUTERS_ANN_05000,2006-11-03,"u.s. probes ford escape, mazda tribute over fires",<NA>,-1,negative,0.649090,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,100,8,49


In [6]:
# ============================================================
# CELL 3 - DATASET KONTROLÜ
# Burada etiketi ve metinleri kontrol et.
# Model yükleme yoktur.
# ============================================================

print("Toplam satır:", len(annotation_all_df))

for col in ["annotation_id", "text_en", "finbert_label", "chatgpt_label", "final_label"]:
    if col in annotation_all_df.columns:
        print(f"{col} boş sayısı:", annotation_all_df[col].isna().sum())

print("\nFinBERT label dağılımı:")
if "finbert_label" in annotation_all_df.columns:
    print(annotation_all_df["finbert_label"].value_counts(dropna=False))

print("\nChatGPT label dağılımı:")
if "chatgpt_label" in annotation_all_df.columns:
    print(annotation_all_df["chatgpt_label"].value_counts(dropna=False))

print("\nFinal label dağılımı:")
if "final_label" in annotation_all_df.columns:
    print(annotation_all_df["final_label"].value_counts(dropna=False))

# Metin uzunluk kontrolü
if "text_en" in annotation_all_df.columns:
    annotation_all_df["text_en"] = annotation_all_df["text_en"].astype("string").str.strip()
    annotation_all_df["n_words_check"] = annotation_all_df["text_en"].str.split().str.len()
    annotation_all_df["text_len_check"] = annotation_all_df["text_en"].str.len()

    print("\nKelime sayısı özeti:")
    display(annotation_all_df["n_words_check"].describe())

    print("\nKarakter uzunluğu özeti:")
    display(annotation_all_df["text_len_check"].describe())

# Uzun gövde yanlışlıkla kalmış mı?
long_texts = annotation_all_df[annotation_all_df.get("text_len_check", pd.Series([], dtype=int)) > 250].copy()
print("\n250 karakter üstü metin sayısı:", len(long_texts))
if len(long_texts) > 0:
    display(long_texts[[c for c in ["annotation_id", "text_en", "text_len_check", "source_file"] if c in long_texts.columns]].head(PREVIEW_ROWS))

# Kontrol için örnekler
print("\nİlk 3 metin:")
for i, row in annotation_all_df.head(PREVIEW_ROWS).iterrows():
    print("=" * 100)
    print(row.get("annotation_id", ""), "| finbert:", row.get("finbert_label", ""), "| final:", row.get("final_label", ""), "| chatgpt:", row.get("chatgpt_label", ""))
    print(row.get("text_en", ""))


Toplam satır: 10000
annotation_id boş sayısı: 0
text_en boş sayısı: 0
finbert_label boş sayısı: 0
chatgpt_label boş sayısı: 5000
final_label boş sayısı: 5000

FinBERT label dağılımı:
finbert_label
positive    3494
negative    3492
neutral     3014
Name: count, dtype: int64[pyarrow]

ChatGPT label dağılımı:
chatgpt_label
<NA>        5000
positive    2275
negative    1579
neutral     1146
Name: count, dtype: int64[pyarrow]

Final label dağılımı:
final_label
<NA>        5000
positive    2275
negative    1579
neutral     1146
Name: count, dtype: int64[pyarrow]

Kelime sayısı özeti:


count    10000.000000
mean         7.298600
std          1.440988
min          4.000000
25%          6.000000
50%          7.000000
75%          8.000000
max         12.000000
Name: n_words_check, dtype: float64


Karakter uzunluğu özeti:


count     10000.0
mean      44.4246
std      6.697612
min          20.0
25%          41.0
50%          46.0
75%          49.0
max          72.0
Name: text_len_check, dtype: Float64


250 karakter üstü metin sayısı: 0

İlk 3 metin:
REUTERS_ANN_00001 | finbert: negative | final: <NA> | chatgpt: <NA>
s.africa watchdog to probe gold fields bid report
REUTERS_ANN_00001 | finbert: negative | final: negative | chatgpt: negative
s.africa watchdog to probe gold fields bid report
REUTERS_ANN_00002 | finbert: neutral | final: <NA> | chatgpt: <NA>
new barbie girls sashay into view with mp-3


In [7]:
# ============================================================
# CELL 4 - EVAL DATAFRAME HAZIRLA
# Gold label: final_label varsa onu kullanır, boşsa chatgpt_label kullanır.
# Model yükleme yoktur.
# ============================================================

df_eval = annotation_all_df.copy()

# Text kolonu
if "text_en" in df_eval.columns:
    df_eval["eval_text"] = df_eval["text_en"]
elif "text" in df_eval.columns:
    df_eval["eval_text"] = df_eval["text"]
else:
    raise ValueError("annotation_all_df içinde text_en veya text kolonu bulunamadı.")

df_eval["eval_text"] = (
    df_eval["eval_text"]
    .astype("string")
    .str.strip()
)

# Gold label: final_label varsa onu kullan, boşsa chatgpt_label kullan
if "final_label" in df_eval.columns:
    df_eval["gold_label"] = df_eval["final_label"]
else:
    df_eval["gold_label"] = pd.NA

df_eval["gold_label"] = (
    df_eval["gold_label"]
    .astype("string")
    .str.lower()
    .str.strip()
    .replace(["", "nan", "none", "<na>", "NaN"], pd.NA)
)

if "chatgpt_label" in df_eval.columns:
    chatgpt_label = (
        df_eval["chatgpt_label"]
        .astype("string")
        .str.lower()
        .str.strip()
        .replace(["", "nan", "none", "<na>", "NaN"], pd.NA)
    )
    df_eval["gold_label"] = df_eval["gold_label"].fillna(chatgpt_label)

# Sadece geçerli eval satırları
df_eval = df_eval[
    df_eval["eval_text"].notna()
    & (df_eval["eval_text"] != "")
    & df_eval["gold_label"].isin(VALID_LABELS)
].copy()

df_eval = df_eval.reset_index(drop=True)
df_eval["gold_id"] = df_eval["gold_label"].map(LABEL2ID).astype(int)

print("Eval shape:", df_eval.shape)

print("\nGold label distribution:")
print(df_eval["gold_label"].value_counts().reindex(VALID_LABELS))

print("\nGold label ratio:")
print(df_eval["gold_label"].value_counts(normalize=True).mul(100).round(2).reindex(VALID_LABELS))

print("\nİlk 3 eval örneği:")
display(df_eval[[c for c in ["annotation_id", "date", "eval_text", "finbert_label", "gold_label"] if c in df_eval.columns]].head(PREVIEW_ROWS))

if len(df_eval) == 0:
    raise ValueError("Eval set boş. final_label veya chatgpt_label kolonlarında geçerli label yok.")


Eval shape: (5000, 23)

Gold label distribution:
gold_label
negative    1579
neutral     1146
positive    2275
Name: count, dtype: int64[pyarrow]

Gold label ratio:
gold_label
negative    31.58
neutral     22.92
positive     45.5
Name: proportion, dtype: double[pyarrow]

İlk 3 eval örneği:


,annotation_id,date,eval_text,finbert_label,gold_label
0,REUTERS_ANN_00001,2007-04-12,s.africa watchdog to probe gold fields bid report,negative,negative
1,REUTERS_ANN_00002,2007-04-26,new barbie girls sashay into view with mp-3,neutral,positive
2,REUTERS_ANN_00003,2006-12-17,"stocks await data, mergers and santa",neutral,neutral


## Buraya kadar dataset yüklendi ve kontrol edildi

Bundan sonraki hücrelerde `torch` ve modeller yüklenir. Eğer PyTorch import hatası alırsan kernel'i restart edip bu notebook'u baştan çalıştır.


In [8]:
# ============================================================
# CELL 5 - TORCH VE TRANSFORMERS IMPORT
# Model yükleme bundan sonra başlar.
# ============================================================

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, logging as hf_logging

hf_logging.set_verbosity_error()

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Torch version: 2.11.0+cpu
CUDA available: False
Device: cpu


In [9]:
# ============================================================
# CELL 6 - METRİK FONKSİYONLARI
# ============================================================

def compute_full_metrics(y_true, y_pred, model_name, test_set_name):
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    precision_macro, recall_macro, _, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    precision_weighted, recall_weighted, _, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    return {
        "model": model_name,
        "test_set": test_set_name,
        "n_eval": int(len(y_true)),
        "accuracy": float(acc),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(precision_weighted),
        "recall_weighted": float(recall_weighted),
        "f1_weighted": float(f1_weighted),
    }


def make_report_and_cm(y_true, y_pred):
    report_text = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=VALID_LABELS,
        digits=4,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2]
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{x}" for x in VALID_LABELS],
        columns=[f"pred_{x}" for x in VALID_LABELS],
    )

    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    cm_norm_df = pd.DataFrame(
        cm_norm,
        index=[f"true_{x}" for x in VALID_LABELS],
        columns=[f"pred_{x}" for x in VALID_LABELS],
    )

    return report_text, cm_df, cm_norm_df


def print_model_result(model_name, y_true, y_pred, test_set_name=TEST_SET_NAME):
    metrics = compute_full_metrics(
        y_true=y_true,
        y_pred=y_pred,
        model_name=model_name,
        test_set_name=test_set_name
    )

    report_text, cm_df, cm_norm_df = make_report_and_cm(y_true, y_pred)

    print("\n" + "=" * 100)
    print(f"{model_name} RESULT")
    print("=" * 100)

    display(pd.DataFrame([metrics]).round(4))

    print("\nClassification Report:")
    print(report_text)

    print("\nConfusion Matrix:")
    display(cm_df)

    print("\nConfusion Matrix Normalized:")
    display(cm_norm_df.round(3))

    return metrics


def print_prediction_details(model_name, model_eval_df):
    model_eval_df = model_eval_df.copy()

    model_eval_df["is_correct"] = model_eval_df["gold_label"] == model_eval_df["pred_label"]

    print("\nPrediction distribution:")
    print(model_eval_df["pred_label"].value_counts().reindex(VALID_LABELS))

    print("\nCorrect / Wrong:")
    print(model_eval_df["is_correct"].value_counts())
    print(model_eval_df["is_correct"].value_counts(normalize=True).mul(100).round(2))

    print("\nAccuracy by true label:")
    display(
        model_eval_df.groupby("gold_label")["is_correct"]
        .agg(["count", "mean"])
        .rename(columns={"mean": "accuracy_by_label"})
        .reindex(VALID_LABELS)
    )

    if "pred_confidence" in model_eval_df.columns:
        model_eval_df["pred_confidence"] = pd.to_numeric(
            model_eval_df["pred_confidence"],
            errors="coerce"
        )

        print("\nConfidence summary:")
        display(model_eval_df["pred_confidence"].describe())

    wrong_df = model_eval_df[model_eval_df["is_correct"] == False].copy()

    if "pred_confidence" in wrong_df.columns:
        wrong_df = wrong_df.sort_values("pred_confidence", ascending=False)

    print("\nWrong predictions:", wrong_df.shape)

    show_cols = [
        "annotation_id",
        "date",
        "eval_text",
        "gold_label",
        "pred_label",
        "pred_confidence",
        "finbert_label",
        "finbert_confidence",
        "chatgpt_reason_tr",
    ]
    show_cols = [c for c in show_cols if c in wrong_df.columns]

    display(wrong_df[show_cols].head(PREVIEW_ROWS))

    return model_eval_df


In [10]:
# ============================================================
# CELL 7 - ORIGINAL FINBERT BASELINE
# Dosyadaki finbert_label kullanılır. Model yüklenmez.
# ============================================================

all_metrics = []
all_prediction_dfs = {}

if "finbert_label" not in df_eval.columns:
    raise ValueError("df_eval içinde finbert_label kolonu yok. Original FinBERT baseline hesaplanamaz.")

finbert_eval_df = df_eval.copy()
finbert_eval_df["pred_label"] = (
    finbert_eval_df["finbert_label"]
    .astype("string")
    .str.lower()
    .str.strip()
)

finbert_eval_df = finbert_eval_df[finbert_eval_df["pred_label"].isin(VALID_LABELS)].copy().reset_index(drop=True)

if "finbert_confidence" in finbert_eval_df.columns:
    finbert_eval_df["pred_confidence"] = pd.to_numeric(finbert_eval_df["finbert_confidence"], errors="coerce")
else:
    finbert_eval_df["pred_confidence"] = np.nan

finbert_eval_df["pred_id"] = finbert_eval_df["pred_label"].map(LABEL2ID).astype(int)

y_true_finbert = finbert_eval_df["gold_id"].astype(int).values
y_pred_finbert = finbert_eval_df["pred_id"].astype(int).values

metrics = print_model_result(
    model_name="original_finbert_file_label",
    y_true=y_true_finbert,
    y_pred=y_pred_finbert,
    test_set_name=TEST_SET_NAME
)

all_metrics.append(metrics)

finbert_eval_df = print_prediction_details(
    model_name="original_finbert_file_label",
    model_eval_df=finbert_eval_df
)

all_prediction_dfs["original_finbert_file_label"] = finbert_eval_df



original_finbert_file_label RESULT


,model,test_set,n_eval,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,original_finbert_file_label,reuters_annotation_5000,5000,0.6966,0.6882,0.7021,0.6866,0.7245,0.6966,0.7016



Classification Report:
              precision    recall  f1-score   support

    negative     0.7125    0.7878    0.7483      1579
     neutral     0.5123    0.6736    0.5820      1146
    positive     0.8397    0.6448    0.7295      2275

    accuracy                         0.6966      5000
   macro avg     0.6882    0.7021    0.6866      5000
weighted avg     0.7245    0.6966    0.7016      5000


Confusion Matrix:


,pred_negative,pred_neutral,pred_positive
true_negative,1244,228,107
true_neutral,201,772,173
true_positive,301,507,1467



Confusion Matrix Normalized:


...

,count,accuracy_by_label
gold_label,,
negative,1579,0.78784
neutral,1146,0.673647
positive,2275,0.644835



Confidence summary:


count    5000.000000
mean        0.825997
std         0.145649
min         0.363184
25%         0.733819
50%         0.881386
75%         0.944828
max         0.977044
Name: pred_confidence, dtype: float64


Wrong predictions: (1517, 27)


,annotation_id,date,eval_text,gold_label,pred_label,pred_confidence,finbert_label,finbert_confidence,chatgpt_reason_tr
673,REUTERS_ANN_00674,2007-04-25,salon operator regis reports quarterly profit,positive,negative,0.976691,negative,0.976691,Kâr açıklanması olumlu; başlıkta düşüş veya beklenti altı sinyali yok.
1657,REUTERS_ANN_01658,2007-01-18,jobless claims post surprising drop last week,positive,negative,0.976435,negative,0.976435,İşgücü piyasasına ilişkin güçlü makro sinyal.
2179,REUTERS_ANN_02180,2006-12-14,"jobless claims fell 20,000 in latest week",positive,negative,0.976320,negative,0.976320,İşsizlik başvurularındaki düşüş işgücü piyasası için olumlu makro sinyal.


In [11]:
# ============================================================
# CELL 8 - FINE-TUNED MODEL PREDICTION FONKSİYONU
# ============================================================

@torch.no_grad()
def predict_with_finetuned_model(model_dir, texts, batch_size=32, max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)

    model.to(device)
    model.eval()

    pred_ids_all = []
    pred_labels = []
    confidences = []

    score_negative = []
    score_neutral = []
    score_positive = []

    texts = [str(x) for x in texts]

    for start in tqdm(range(0, len(texts), batch_size), desc=f"Predicting {Path(model_dir).parent.name}"):
        batch_texts = texts[start:start + batch_size]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=-1).detach().cpu().numpy()
        pred_ids = probs.argmax(axis=1)

        for pred_id, prob_vec in zip(pred_ids, probs):
            pred_id = int(pred_id)

            pred_ids_all.append(pred_id)
            pred_labels.append(ID2LABEL[pred_id])
            confidences.append(float(prob_vec[pred_id]))

            score_negative.append(float(prob_vec[0]))
            score_neutral.append(float(prob_vec[1]))
            score_positive.append(float(prob_vec[2]))

    del model
    del tokenizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

    return pd.DataFrame({
        "pred_id": pred_ids_all,
        "pred_label": pred_labels,
        "pred_confidence": confidences,
        "score_negative": score_negative,
        "score_neutral": score_neutral,
        "score_positive": score_positive,
    })


In [12]:
# ============================================================
# CELL 9 - FINE-TUNED MODELLERİ SIRAYLA TEST ET
# ============================================================

for run in MODEL_RUNS:
    run_name = run["run_name"]
    model_dir = Path(run["model_dir"])

    if not model_dir.exists():
        print("\nUYARI: Model klasörü bulunamadı:", model_dir)
        continue

    print("\n" + "#" * 120)
    print(f"MODEL TEST EDİLİYOR: {run_name}")
    print("Model dir:", model_dir)
    print("#" * 120)

    pred_df = predict_with_finetuned_model(
        model_dir=model_dir,
        texts=df_eval["eval_text"].tolist(),
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH
    )

    y_true = df_eval["gold_id"].astype(int).values
    y_pred = pred_df["pred_id"].astype(int).values

    metrics = print_model_result(
        model_name=run_name,
        y_true=y_true,
        y_pred=y_pred,
        test_set_name=TEST_SET_NAME
    )

    all_metrics.append(metrics)

    base_cols = [
        "annotation_id",
        "date",
        "source_file",
        "batch_no",
        "eval_text",
        "gold_label",
        "gold_id",
        "finbert_label",
        "finbert_confidence",
        "chatgpt_reason_tr",
    ]
    base_cols = [c for c in base_cols if c in df_eval.columns]

    model_eval_df = pd.concat(
        [
            df_eval[base_cols].reset_index(drop=True),
            pred_df.reset_index(drop=True),
        ],
        axis=1
    )

    model_eval_df = print_prediction_details(
        model_name=run_name,
        model_eval_df=model_eval_df
    )

    all_prediction_dfs[run_name] = model_eval_df



########################################################################################################################
MODEL TEST EDİLİYOR: roberta_base
Model dir: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\roberta_base\final_model
########################################################################################################################


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Predicting roberta_base:   0%|          | 0/157 [00:00<?, ?it/s]


roberta_base RESULT


,model,test_set,n_eval,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,roberta_base,reuters_annotation_5000,5000,0.6268,0.7464,0.6723,0.6388,0.802,0.6268,0.6486


...

,count,accuracy_by_label
gold_label,,
negative,1579,0.533249
neutral,1146,0.866492
positive,2275,0.460659



Confidence summary:


count    5000.000000
mean        0.956633
std         0.097915
min         0.393685
25%         0.980879
50%         0.995833
75%         0.997650
max         0.998500
Name: pred_confidence, dtype: float64


Wrong predictions: (2117, 16)


,annotation_id,date,eval_text,gold_label,pred_label,pred_confidence,finbert_label,finbert_confidence,chatgpt_reason_tr
3316,REUTERS_ANN_03317,2006-11-01,cingular to launch cell phone music service -report,positive,neutral,0.998467,positive,0.777344,Yeni müzik hizmeti gelir çeşitlendirmesi ve müşteri bağlılığı potansiyeli taşır.
1427,REUTERS_ANN_01428,2006-12-19,harrahs to be acquired for $16.7 billion paper,positive,neutral,0.998464,neutral,0.653342,Satın alma teklifi hedef şirket için genelde olumlu değerleme sinyali taşır.
136,REUTERS_ANN_00137,2006-11-20,factbox outline of nasdaqs new bid for lse,positive,neutral,0.998461,positive,0.733128,Yeni teklif M&A değeri ve hedef hissedarları için potansiyel prim sinyali verir.


In [14]:
# ============================================================
# CELL 10 - FINAL SUMMARY
# ============================================================

summary_df = pd.DataFrame(all_metrics)

if summary_df.empty:
    print("Hiçbir model başarıyla test edilemedi.")
else:
    summary_df = summary_df.sort_values(
        "f1_macro",
        ascending=False
    ).reset_index(drop=True)

    print("\n" + "=" * 120)
    print("FINAL SUMMARY ON REUTERS ANNOTATION 5000 SET")
    print("=" * 120)

    display(summary_df.round(4))

    print("\nMarkdown tablo:")
    print(summary_df.round(4).to_markdown(index=False))

    best = summary_df.iloc[0]

    print("\nBest model:")
    print("model:", best["model"])
    print("accuracy:", round(float(best["accuracy"]), 4))
    print("f1_macro:", round(float(best["f1_macro"]), 4))
    print("f1_weighted:", round(float(best["f1_weighted"]), 4))



FINAL SUMMARY ON REUTERS ANNOTATION 5000 SET


,model,test_set,n_eval,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,original_finbert_file_label,reuters_annotation_5000,5000,0.6966,0.6882,0.7021,0.6866,0.7245,0.6966,0.7016
1,roberta_base,reuters_annotation_5000,5000,0.6268,0.7464,0.6723,0.6388,0.8020,0.6268,0.6486
2,bert_base_uncased,reuters_annotation_5000,5000,0.6034,0.7165,0.6451,0.6131,0.7686,0.6034,0.6240
3,distilbert_base_uncased,reuters_annotation_5000,5000,0.5766,0.6883,0.6201,0.5869,0.7391,0.5766,0.5953



Markdown tablo:
| model                       | test_set                |   n_eval |   accuracy |   precision_macro |   recall_macro |   f1_macro |   precision_weighted |   recall_weighted |   f1_weighted |
|:----------------------------|:------------------------|---------:|-----------:|------------------:|---------------:|-----------:|---------------------:|------------------:|--------------:|
| original_finbert_file_label | reuters_annotation_5000 |     5000 |     0.6966 |            0.6882 |         0.7021 |     0.6866 |               0.7245 |            0.6966 |        0.7016 |
| roberta_base                | reuters_annotation_5000 |     5000 |     0.6268 |            0.7464 |         0.6723 |     0.6388 |               0.802  |            0.6268 |        0.6486 |
| bert_base_uncased           | reuters_annotation_5000 |     5000 |     0.6034 |            0.7165 |         0.6451 |     0.6131 |               0.7686 |            0.6034 |        0.624  |
| distilbert_base_uncased   